<a href="https://colab.research.google.com/github/afrineejhilik71-arch/plant-disease-detection/blob/main/02_ResNet50_Baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

print(os.listdir('/content/drive/MyDrive'))

['Getting started.pdf', 'image1.PNG', 'Sayada Zakia Afrine blacknew.pdf', 'Sayada Zakia Afrine blacknew.gdoc', 'Class 4 EV science OK.doc', 'Class 4 EV science OK.gdoc', 'Norway', 'Residence Card Zakia.pdf', 'prove-kap-1-3.pdf', 'prove-kap-1-3 -----Sayada Zakia Afrinee.gdoc', 'Adobe Scan 21 Aug 2023_1.jpg', '16CnLeQpLdW3P9V9Pc2MZHmtw0kGUJvcf', 'Cover Letter - Rystad Energy.gdoc', 'Study Jhilik', 'Group 7 Project Writing My Part.gdoc', 'Admission', 'Colab Notebooks', 'dataset', 'TCS Media Release Form (Externals over 16yrs) INNO-SCI 2026.pdf', 'international-airline-passengers.csv', 'TestScript_LSTM.ipynb', 'DATA_Test.csv']


In [3]:
import os

for name in os.listdir('/content/drive/MyDrive'):
    path = os.path.join('/content/drive/MyDrive', name)
    if os.path.isdir(path):
        print("Folder:", name)

Folder: Norway
Folder: Study Jhilik
Folder: Admission
Folder: Colab Notebooks
Folder: dataset


In [4]:
import os

dataset_path = '/content/drive/MyDrive/dataset'

for name in os.listdir(dataset_path):
    path = os.path.join(dataset_path, name)

    if os.path.isdir(path):
        print("Folder:", name)
    else:
        print("File:", name)

Folder: val
Folder: train
Folder: AgriVision4_Tomato
File: tomato_conflicting_labels_review.csv
File: tomato_baseline_split.csv
File: resnet50_tomato_pilot_epoch1.pth
File: resnet50_tomato_best_finetuned.pth
File: resnet50_tomato_finetuning_history.csv
File: efficientnet_b0_tomato_best_finetuned.pth
File: efficientnet_b0_tomato_finetuning_history.csv
File: densenet121_tomato_finetuning_history.csv
File: densenet121_tomato_best_finetuned.pth
File: tomato_baseline_results.csv


In [5]:
import os

dataset_path = '/content/drive/MyDrive/dataset'

for split in ['train', 'val', 'test']:
    split_path = os.path.join(dataset_path, split)

    if os.path.isdir(split_path):
        print(f'\n{split.upper()} folders:')
        print(os.listdir(split_path)[:30])
    else:
        print(f'\n{split.upper()}: folder not found')


TRAIN folders:
['not', 'alpaca']

VAL folders:
['alpaca', 'not']

TEST: folder not found


In [6]:
import os

root = '/content/drive/MyDrive'

for current, folders, files in os.walk(root):
    folders[:] = [f for f in folders if f != '.ipynb_checkpoints']

    for folder in folders:
        if any(word in folder.lower()
               for word in ['agri', 'tomato', 'papaya', 'zucchini', 'gourd']):
            print(os.path.join(current, folder))

    # Avoid searching too deeply for now
    if current[len(root):].count(os.sep) >= 2:
        folders[:] = []

/content/drive/MyDrive/dataset/AgriVision4_Tomato


In [7]:
import os

tomato_path = '/content/drive/MyDrive/dataset/AgriVision4_Tomato'

print("Folder exists:", os.path.isdir(tomato_path))

for class_name in sorted(os.listdir(tomato_path)):
    class_path = os.path.join(tomato_path, class_name)

    if os.path.isdir(class_path):
        images = [
            f for f in os.listdir(class_path)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ]
        print(f"{class_name}: {len(images)} images")

Folder exists: True
Downy Mildew: 57 images
Healthy: 288 images
Mosaic: 195 images
Spot: 311 images
white_spot: 65 images


In [8]:
from collections import defaultdict
from pathlib import Path
import hashlib

tomato_path = Path('/content/drive/MyDrive/dataset/AgriVision4_Tomato')

hash_to_files = defaultdict(list)

for class_dir in sorted(tomato_path.iterdir()):
    if not class_dir.is_dir():
        continue

    for img_path in class_dir.iterdir():
        if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png']:
            digest = hashlib.sha256(img_path.read_bytes()).hexdigest()
            hash_to_files[digest].append((class_dir.name, img_path.name))

duplicate_groups = [
    files for files in hash_to_files.values()
    if len(files) > 1
]

conflicting_groups = [
    files for files in duplicate_groups
    if len(set(label for label, _ in files)) > 1
]

print("Unique image contents:", len(hash_to_files))
print("Exact duplicate groups:", len(duplicate_groups))
print("Duplicates across different classes:", len(conflicting_groups))

Unique image contents: 804
Exact duplicate groups: 112
Duplicates across different classes: 9


In [9]:
for files in conflicting_groups:
    print("\nConflicting group:")
    for label, filename in files:
        print(f"  Class: {label} | File: {filename}")


Conflicting group:
  Class: Downy Mildew | File: IMG_20251231_151814(1).png
  Class: Spot | File: IMG_20251231_151814.png

Conflicting group:
  Class: Healthy | File: IMG_20251231_151236(1).png
  Class: Spot | File: IMG_20251231_151236.png

Conflicting group:
  Class: Healthy | File: IMG_20251231_151306.png
  Class: Mosaic | File: IMG_20251231_151306(1).png

Conflicting group:
  Class: Healthy | File: IMG_20251231_151324(1).png
  Class: Spot | File: IMG_20251231_151324.png

Conflicting group:
  Class: Healthy | File: IMG_20251231_151459.png
  Class: Mosaic | File: IMG_20251231_151459(1).png

Conflicting group:
  Class: Healthy | File: IMG_20251231_151518.png
  Class: Mosaic | File: IMG_20251231_151518(1).png

Conflicting group:
  Class: Healthy | File: IMG_20251231_151521.png
  Class: Spot | File: IMG_20251231_151521(1).png

Conflicting group:
  Class: Healthy | File: IMG_20251231_151826.png
  Class: Spot | File: IMG_20251231_151826(1).png

Conflicting group:
  Class: Healthy | File: 

In [10]:
from collections import Counter

conflict_class_counts = Counter()

for files in conflicting_groups:
    labels = sorted(set(label for label, _ in files))
    print("\nConflicting labels:", " vs ".join(labels))

    for label, filename in files:
        conflict_class_counts[label] += 1
        print(f"  {label}: {filename}")

print("\nAffected files per class:")
for label, count in sorted(conflict_class_counts.items()):
    print(f"{label}: {count}")


Conflicting labels: Downy Mildew vs Spot
  Downy Mildew: IMG_20251231_151814(1).png
  Spot: IMG_20251231_151814.png

Conflicting labels: Healthy vs Spot
  Healthy: IMG_20251231_151236(1).png
  Spot: IMG_20251231_151236.png

Conflicting labels: Healthy vs Mosaic
  Healthy: IMG_20251231_151306.png
  Mosaic: IMG_20251231_151306(1).png

Conflicting labels: Healthy vs Spot
  Healthy: IMG_20251231_151324(1).png
  Spot: IMG_20251231_151324.png

Conflicting labels: Healthy vs Mosaic
  Healthy: IMG_20251231_151459.png
  Mosaic: IMG_20251231_151459(1).png

Conflicting labels: Healthy vs Mosaic
  Healthy: IMG_20251231_151518.png
  Mosaic: IMG_20251231_151518(1).png

Conflicting labels: Healthy vs Spot
  Healthy: IMG_20251231_151521.png
  Spot: IMG_20251231_151521(1).png

Conflicting labels: Healthy vs Spot
  Healthy: IMG_20251231_151826.png
  Spot: IMG_20251231_151826(1).png

Conflicting labels: Healthy vs Mosaic
  Healthy: IMG_20251231_151906.png
  Mosaic: IMG_20251231_151906(1).png

Affected f

In [11]:
from pathlib import Path

tomato_path = Path(
    '/content/drive/MyDrive/dataset/AgriVision4_Tomato'
)

for class_dir in sorted(tomato_path.iterdir()):
    if class_dir.is_dir():
        count = sum(
            1 for p in class_dir.iterdir()
            if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}
        )
        print(f'{class_dir.name}: {count}')

Downy Mildew: 57
Healthy: 288
Mosaic: 195
Spot: 311
white_spot: 65


In [12]:
import sklearn
print("scikit-learn version:", sklearn.__version__)

scikit-learn version: 1.6.1


In [13]:
from pathlib import Path
import pandas as pd

tomato_path = Path(
    '/content/drive/MyDrive/dataset/AgriVision4_Tomato'
)

records = []

for class_dir in sorted(tomato_path.iterdir()):
    if class_dir.is_dir():
        for img_path in sorted(class_dir.iterdir()):
            if img_path.suffix.lower() in {'.jpg', '.jpeg', '.png'}:
                records.append({
                    'path': str(img_path),
                    'class': class_dir.name
                })

df = pd.DataFrame(records)

print("Total images:", len(df))
print("\nImages per class:")
print(df['class'].value_counts().sort_index())
print("\nFirst five records:")
display(df.head())

Total images: 916

Images per class:
class
Downy Mildew     57
Healthy         288
Mosaic          195
Spot            311
white_spot       65
Name: count, dtype: int64

First five records:


,path,class
0,/content/drive/MyDrive/dataset/AgriVision4_Tom...,Downy Mildew
1,/content/drive/MyDrive/dataset/AgriVision4_Tom...,Downy Mildew
2,/content/drive/MyDrive/dataset/AgriVision4_Tom...,Downy Mildew
3,/content/drive/MyDrive/dataset/AgriVision4_Tom...,Downy Mildew
4,/content/drive/MyDrive/dataset/AgriVision4_Tom...,Downy Mildew


In [14]:
import hashlib

def get_sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

df['sha256'] = df['path'].apply(get_sha256)

print("Total files:", len(df))
print("Unique image contents:", df['sha256'].nunique())
print("Duplicate files beyond first copies:",
      len(df) - df['sha256'].nunique())

Total files: 916
Unique image contents: 804
Duplicate files beyond first copies: 112


In [15]:
labels_per_hash = df.groupby('sha256')['class'].nunique()

print("Unique image contents:", len(labels_per_hash))
print("Groups with multiple class labels:",
      (labels_per_hash > 1).sum())

print("\nConflicting label combinations:")
conflicts = (
    df.groupby('sha256')['class']
      .agg(lambda x: sorted(set(x)))
)
print(conflicts[conflicts.apply(len) > 1].to_string())

Unique image contents: 804
Groups with multiple class labels: 9

Conflicting label combinations:
sha256
2ad31356a5195317d3958dc903d77b3f5b734373a24f7dc28b9e906d638baeb9       [Healthy, Mosaic]
45c3a05cef8ba743647bc9bb33b2f781da98eb4d1f0617d23a8052a03339d604       [Healthy, Mosaic]
7397f5706e99c1ee227d9668daf13d7279fb729fde2356a1d971f540077c4793         [Healthy, Spot]
7a8b455636666106d1ad012bc134b8930554e1621fe43a1c1345129cd7d004cc       [Healthy, Mosaic]
8de6ac67afe65b67b3baac3de441c576f7a60bce17c9c17fa6969598e4b6454a         [Healthy, Spot]
a00a66d13a81023168ca7820aebc5a4ad9af5a090ac41f374cbccdccf30478c2    [Downy Mildew, Spot]
bef2941eb27f8c0debf1eec0d8968ff19ebd9c562affcf0befd8583a4b17c136         [Healthy, Spot]
e001456ebf27032769bd28d84ac6ca459bd2d6fe124a8fca4db6d1ecf0ada88e         [Healthy, Spot]
fd430cee957dfaed3f2e8c557dc36439ae384951b8e42593406b5872d5977749       [Healthy, Mosaic]


In [16]:
conflict_hashes = labels_per_hash[labels_per_hash > 1].index

conflict_df = df[df['sha256'].isin(conflict_hashes)].copy()

conflict_df = conflict_df.sort_values(['sha256', 'class'])

review_path = (
    '/content/drive/MyDrive/dataset/'
    'tomato_conflicting_labels_review.csv'
)

conflict_df.to_csv(review_path, index=False)

print("Conflicting image records:", len(conflict_df))
print("Report saved to:", review_path)
display(conflict_df[['class', 'path', 'sha256']].head(10))

Conflicting image records: 18
Report saved to: /content/drive/MyDrive/dataset/tomato_conflicting_labels_review.csv


,class,path,sha256
196,Healthy,/content/drive/MyDrive/dataset/AgriVision4_Tom...,2ad31356a5195317d3958dc903d77b3f5b734373a24f7d...
507,Mosaic,/content/drive/MyDrive/dataset/AgriVision4_Tom...,2ad31356a5195317d3958dc903d77b3f5b734373a24f7d...
239,Healthy,/content/drive/MyDrive/dataset/AgriVision4_Tom...,45c3a05cef8ba743647bc9bb33b2f781da98eb4d1f0617...
509,Mosaic,/content/drive/MyDrive/dataset/AgriVision4_Tom...,45c3a05cef8ba743647bc9bb33b2f781da98eb4d1f0617...
240,Healthy,/content/drive/MyDrive/dataset/AgriVision4_Tom...,7397f5706e99c1ee227d9668daf13d7279fb729fde2356...
728,Spot,/content/drive/MyDrive/dataset/AgriVision4_Tom...,7397f5706e99c1ee227d9668daf13d7279fb729fde2356...
270,Healthy,/content/drive/MyDrive/dataset/AgriVision4_Tom...,7a8b455636666106d1ad012bc134b8930554e1621fe43a...
510,Mosaic,/content/drive/MyDrive/dataset/AgriVision4_Tom...,7a8b455636666106d1ad012bc134b8930554e1621fe43a...
188,Healthy,/content/drive/MyDrive/dataset/AgriVision4_Tom...,8de6ac67afe65b67b3baac3de441c576f7a60bce17c9c1...
695,Spot,/content/drive/MyDrive/dataset/AgriVision4_Tom...,8de6ac67afe65b67b3baac3de441c576f7a60bce17c9c1...


In [17]:
# Exclude image contents that have conflicting class labels.
conflict_hashes = set(
    labels_per_hash[labels_per_hash > 1].index
)

clean_df = df[~df['sha256'].isin(conflict_hashes)].copy()

print("Original image records:", len(df))
print("Conflicting image records excluded:", len(df) - len(clean_df))
print("Remaining image records:", len(clean_df))
print("Unique image contents remaining:", clean_df['sha256'].nunique())

print("\nRemaining images per class:")
print(clean_df['class'].value_counts().sort_index())

Original image records: 916
Conflicting image records excluded: 18
Remaining image records: 898
Unique image contents remaining: 795

Remaining images per class:
class
Downy Mildew     56
Healthy         280
Mosaic          191
Spot            306
white_spot       65
Name: count, dtype: int64


In [18]:
from sklearn.model_selection import GroupShuffleSplit

# One row per image record; identical hashes stay in one group.
clean_df = clean_df.reset_index(drop=True)

# First: reserve approximately 15% for testing.
split1 = GroupShuffleSplit(
    n_splits=1, test_size=0.15, random_state=42
)

trainval_idx, test_idx = next(
    split1.split(clean_df, groups=clean_df['sha256'])
)

trainval_df = clean_df.iloc[trainval_idx].copy()
test_df = clean_df.iloc[test_idx].copy()

# Second: reserve approximately 15% of the full dataset for validation.
validation_fraction = 0.15 / 0.85

split2 = GroupShuffleSplit(
    n_splits=1,
    test_size=validation_fraction,
    random_state=42
)

train_idx, val_idx = next(
    split2.split(trainval_df, groups=trainval_df['sha256'])
)

train_df = trainval_df.iloc[train_idx].copy()
val_df = trainval_df.iloc[val_idx].copy()

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 630
Validation: 136
Test: 132


In [19]:
# Check class counts in each subset
print("TRAIN class counts:")
print(train_df['class'].value_counts().sort_index())

print("\nVALIDATION class counts:")
print(val_df['class'].value_counts().sort_index())

print("\nTEST class counts:")
print(test_df['class'].value_counts().sort_index())

# Check whether identical image contents cross subsets
train_hashes = set(train_df['sha256'])
val_hashes = set(val_df['sha256'])
test_hashes = set(test_df['sha256'])

print("\nDuplicate leakage checks:")
print("Train vs validation:", len(train_hashes & val_hashes))
print("Train vs test:", len(train_hashes & test_hashes))
print("Validation vs test:", len(val_hashes & test_hashes))

TRAIN class counts:
class
Downy Mildew     38
Healthy         195
Mosaic          135
Spot            221
white_spot       41
Name: count, dtype: int64

VALIDATION class counts:
class
Downy Mildew    10
Healthy         40
Mosaic          28
Spot            45
white_spot      13
Name: count, dtype: int64

TEST class counts:
class
Downy Mildew     8
Healthy         45
Mosaic          28
Spot            40
white_spot      11
Name: count, dtype: int64

Duplicate leakage checks:
Train vs validation: 0
Train vs test: 0
Validation vs test: 0


In [20]:
import pandas as pd

train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

train_df['split'] = 'train'
val_df['split'] = 'validation'
test_df['split'] = 'test'

split_df = pd.concat(
    [train_df, val_df, test_df],
    ignore_index=True
)

split_path = (
    '/content/drive/MyDrive/dataset/'
    'tomato_baseline_split.csv'
)

split_df.to_csv(split_path, index=False)

print("Split saved to:", split_path)
print("Total records:", len(split_df))
print(split_df['split'].value_counts())

Split saved to: /content/drive/MyDrive/dataset/tomato_baseline_split.csv
Total records: 898
split
train         630
validation    136
test          132
Name: count, dtype: int64


In [21]:
import torch
import torchvision
from torchvision import models

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)

print("ResNet50 available:", hasattr(models, "resnet50"))
print("EfficientNet-B0 available:", hasattr(models, "efficientnet_b0"))
print("DenseNet121 available:", hasattr(models, "densenet121"))

print("GPU available:", torch.cuda.is_available())

PyTorch: 2.11.0+cu130
Torchvision: 0.26.0+cu130
ResNet50 available: True
EfficientNet-B0 available: True
DenseNet121 available: True
GPU available: True


In [22]:
import torch

print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu130
GPU available: True
GPU name: Tesla T4


In [23]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [24]:
import pandas as pd
from pathlib import Path

split_path = Path(
    '/content/drive/MyDrive/dataset/tomato_baseline_split.csv'
)

split_df = pd.read_csv(split_path)

print("Split file exists:", split_path.exists())
print("Total records:", len(split_df))
print("\nRecords per split:")
print(split_df['split'].value_counts())
print("\nColumns:", split_df.columns.tolist())

Split file exists: True
Total records: 898

Records per split:
split
train         630
validation    136
test          132
Name: count, dtype: int64

Columns: ['path', 'class', 'sha256', 'split']


In [25]:
from pathlib import Path

missing = [
    path for path in split_df['path']
    if not Path(path).is_file()
]

print("Total records:", len(split_df))
print("Missing image files:", len(missing))

if missing:
    print("\nFirst missing paths:")
    print(*missing[:5], sep="\n")

Total records: 898
Missing image files: 0


In [26]:
from PIL import Image

readable = 0
unreadable = []

for path in split_df['path']:
    try:
        with Image.open(path) as img:
            img.verify()
        readable += 1
    except Exception as e:
        unreadable.append((path, str(e)))

print("Readable images:", readable)
print("Unreadable images:", len(unreadable))

if unreadable:
    print("\nFirst unreadable files:")
    for path, error in unreadable[:5]:
        print(path, error)

Readable images: 898
Unreadable images: 0


In [27]:
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print("GPU:", gpu.name)
    print(f"Total GPU memory: {gpu.total_memory / 1024**3:.2f} GB")
else:
    print("GPU is not available.")

GPU: Tesla T4
Total GPU memory: 14.56 GB


In [28]:
import torch
from torchvision import models, transforms

# Use the GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load pretrained ResNet50
weights = models.ResNet50_Weights.DEFAULT
model = models.resnet50(weights=weights)

# Replace the final layer for our 5 tomato classes
num_features = model.fc.in_features
model.fc = torch.nn.Linear(num_features, 5)

model = model.to(device)

# Use the preprocessing recommended for these pretrained weights
preprocess = weights.transforms()

print("Device:", device)
print("Model: ResNet50")
print("Number of output classes:", model.fc.out_features)
print("Input size:", preprocess.crop_size)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 196MB/s]


Device: cuda
Model: ResNet50
Number of output classes: 5
Input size: [224]


In [29]:
import pandas as pd
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Load the saved experiment split
split_path = '/content/drive/MyDrive/dataset/tomato_baseline_split.csv'
split_df = pd.read_csv(split_path)

# Define a consistent class-to-number mapping
classes = sorted(split_df['class'].unique())
class_to_idx = {name: idx for idx, name in enumerate(classes)}

print("Class mapping:")
print(class_to_idx)

Class mapping:
{'Downy Mildew': 0, 'Healthy': 1, 'Mosaic': 2, 'Spot': 3, 'white_spot': 4}


In [30]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
import torch

# Standard ImageNet preprocessing for pretrained ResNet50
image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class TomatoDataset(Dataset):
    def __init__(self, dataframe, class_to_idx, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.class_to_idx = class_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        image = Image.open(row['path']).convert('RGB')
        label = self.class_to_idx[row['class']]

        if self.transform:
            image = self.transform(image)

        return image, label

print("Dataset loader class created successfully.")

Dataset loader class created successfully.


In [31]:
# Separate the saved split
train_df = split_df[split_df['split'] == 'train'].copy()
val_df = split_df[split_df['split'] == 'validation'].copy()
test_df = split_df[split_df['split'] == 'test'].copy()

# Create datasets
train_dataset = TomatoDataset(train_df, class_to_idx, image_transform)
val_dataset = TomatoDataset(val_df, class_to_idx, image_transform)
test_dataset = TomatoDataset(test_df, class_to_idx, image_transform)

# Create data loaders
batch_size = 16

train_loader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True, num_workers=2
)
val_loader = DataLoader(
    val_dataset, batch_size=batch_size, shuffle=False, num_workers=2
)
test_loader = DataLoader(
    test_dataset, batch_size=batch_size, shuffle=False, num_workers=2
)

print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Test images:", len(test_dataset))

Training images: 630
Validation images: 136
Test images: 132


In [32]:
images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Labels in this batch:", labels.tolist())
print("Image tensor type:", images.dtype)
print("Device available:", device)

Image batch shape: torch.Size([16, 3, 224, 224])
Label batch shape: torch.Size([16])
Labels in this batch: [3, 4, 4, 0, 1, 1, 3, 0, 3, 3, 1, 1, 2, 1, 2, 3]
Image tensor type: torch.float32
Device available: cuda


In [33]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score, f1_score

# Move the model to the GPU
model = model.to(device)

# Train the newly added classification layer first
for param in model.parameters():
    param.requires_grad = False

for param in model.fc.parameters():
    param.requires_grad = True

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

# One training epoch
model.train()
train_loss = 0.0
train_true, train_pred = [], []

for images, labels in train_loader:
    images, labels = images.to(device), labels.to(device)

    optimizer.zero_grad()
    outputs = model(images)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

    train_loss += loss.item() * images.size(0)
    train_true.extend(labels.cpu().tolist())
    train_pred.extend(outputs.argmax(dim=1).cpu().tolist())

train_loss /= len(train_dataset)

# Validation
model.eval()
val_loss = 0.0
val_true, val_pred = [], []

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        val_loss += loss.item() * images.size(0)
        val_true.extend(labels.cpu().tolist())
        val_pred.extend(outputs.argmax(dim=1).cpu().tolist())

val_loss /= len(val_dataset)

print(f"Training loss:   {train_loss:.4f}")
print(f"Validation loss: {val_loss:.4f}")
print(f"Validation accuracy: {accuracy_score(val_true, val_pred):.4f}")
print(f"Validation macro-F1: {f1_score(val_true, val_pred, average='macro', zero_division=0):.4f}")

Training loss:   1.3900
Validation loss: 1.3482
Validation accuracy: 0.4926
Validation macro-F1: 0.3074


In [ ]:
import torch
from pathlib import Path

checkpoint_path = (
    '/content/drive/MyDrive/dataset/'
    'resnet50_tomato_pilot_epoch1.pth'
)

torch.save({
    'model_state_dict': model.state_dict(),
    'class_to_idx': class_to_idx,
    'epoch': 1,
    'train_loss': train_loss,
    'val_loss': val_loss,
    'val_accuracy': accuracy_score(val_true, val_pred),
    'val_macro_f1': f1_score(
        val_true, val_pred, average='macro', zero_division=0
    ),
}, checkpoint_path)

print("Checkpoint saved:", Path(checkpoint_path).exists())
print("Saved to:", checkpoint_path)

In [ ]:
import pandas as pd
from pathlib import Path

results_path = (
    '/content/drive/MyDrive/dataset/'
    'tomato_baseline_results.csv'
)

pilot_result = pd.DataFrame([{
    'crop': 'Tomato',
    'model': 'ResNet50',
    'experiment': 'Pilot - frozen backbone',
    'epoch': 1,
    'train_loss': train_loss,
    'validation_loss': val_loss,
    'validation_accuracy': accuracy_score(val_true, val_pred),
    'validation_macro_f1': f1_score(
        val_true, val_pred, average='macro', zero_division=0
    ),
    'checkpoint': 'resnet50_tomato_pilot_epoch1.pth'
}])

if Path(results_path).exists():
    old_results = pd.read_csv(results_path)
    results = pd.concat([old_results, pilot_result], ignore_index=True)
else:
    results = pilot_result

results.to_csv(results_path, index=False)

print("Results saved:", results_path)
display(results)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    val_true,
    val_pred,
    labels=list(range(len(class_to_idx))),
    target_names=[name for name, idx in sorted(
        class_to_idx.items(), key=lambda item: item[1]
    )],
    zero_division=0
))

In [ ]:
from collections import Counter

class_names = [
    name for name, idx in sorted(
        class_to_idx.items(), key=lambda item: item[1]
    )
]

print("Actual validation labels:")
print(Counter(class_names[label] for label in val_true))

print("\nPredicted validation labels:")
print(Counter(class_names[label] for label in val_pred))

In [ ]:
import torch

print("GPU available:", torch.cuda.is_available())
print("Model available:", 'model' in globals())
print("Training loader available:", 'train_loader' in globals())
print("Validation loader available:", 'val_loader' in globals())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

dataset_dir = Path('/content/drive/MyDrive/dataset')

files_to_check = [
    'tomato_baseline_split.csv',
    'resnet50_tomato_pilot_epoch1.pth',
    'tomato_baseline_results.csv'
]

for filename in files_to_check:
    path = dataset_dir / filename
    print(f"{filename}: {'Found' if path.is_file() else 'Not found'}")

In [ ]:
import pandas as pd

dataset_dir = '/content/drive/MyDrive/dataset'

split_df = pd.read_csv(
    f'{dataset_dir}/tomato_baseline_split.csv'
)

classes = sorted(split_df['class'].unique())
class_to_idx = {name: idx for idx, name in enumerate(classes)}

train_df = split_df[split_df['split'] == 'train'].copy()
val_df = split_df[split_df['split'] == 'validation'].copy()
test_df = split_df[split_df['split'] == 'test'].copy()

print("Total records:", len(split_df))
print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Testing:", len(test_df))
print("\nClass mapping:", class_to_idx)

In [ ]:
import torch
from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class TomatoDataset(Dataset):
    def __init__(self, dataframe, class_to_idx, transform):
        self.dataframe = dataframe.reset_index(drop=True)
        self.class_to_idx = class_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        with Image.open(row['path']) as img:
            image = img.convert('RGB')

        image = self.transform(image)
        label = self.class_to_idx[row['class']]

        return image, label

train_dataset = TomatoDataset(train_df, class_to_idx, image_transform)
val_dataset = TomatoDataset(val_df, class_to_idx, image_transform)
test_dataset = TomatoDataset(test_df, class_to_idx, image_transform)

train_loader = DataLoader(
    train_dataset, batch_size=16, shuffle=True, num_workers=2
)
val_loader = DataLoader(
    val_dataset, batch_size=16, shuffle=False, num_workers=2
)
test_loader = DataLoader(
    test_dataset, batch_size=16, shuffle=False, num_workers=2
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

In [ ]:
import torch
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint_path = (
    '/content/drive/MyDrive/dataset/'
    'resnet50_tomato_pilot_epoch1.pth'
)

# Recreate the same ResNet50 architecture
model = models.resnet50(weights=None)
model.fc = torch.nn.Linear(model.fc.in_features, len(class_to_idx))

# Load the saved pilot weights
checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=True
)

model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)

print("Checkpoint restored successfully.")
print("Device:", device)
print("Saved epoch:", checkpoint['epoch'])
print("Saved validation accuracy:", checkpoint['val_accuracy'])
print("Saved validation macro-F1:", checkpoint['val_macro_f1'])

In [ ]:
import copy
import torch
import torch.nn as nn
import torch.optim as optim

# Keep the restored pilot unchanged
finetune_model = copy.deepcopy(model).to(device)

# Unfreeze the pretrained backbone for fine-tuning
for param in finetune_model.parameters():
    param.requires_grad = True

# Use a smaller learning rate for pretrained weights
criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    finetune_model.parameters(),
    lr=1e-5,
    weight_decay=1e-4
)

print("Fine-tuning model prepared.")
print("Trainable parameters:",
      sum(p.numel() for p in finetune_model.parameters()
          if p.requires_grad))
print("Learning rate: 0.00001")

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()

    total_loss = 0.0
    all_true, all_pred = [], []

    with torch.set_grad_enabled(training):
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            if training:
                optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            if training:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * images.size(0)
            all_true.extend(labels.cpu().tolist())
            all_pred.extend(outputs.argmax(dim=1).cpu().tolist())

    average_loss = total_loss / len(loader.dataset)
    accuracy = accuracy_score(all_true, all_pred)
    macro_f1 = f1_score(
        all_true, all_pred, average='macro', zero_division=0
    )

    return average_loss, accuracy, macro_f1, all_true, all_pred


train_loss_ft, train_acc_ft, train_f1_ft, _, _ = run_epoch(
    finetune_model, train_loader, criterion, optimizer
)

val_loss_ft, val_acc_ft, val_f1_ft, val_true_ft, val_pred_ft = run_epoch(
    finetune_model, val_loader, criterion
)

print(f"Training loss: {train_loss_ft:.4f}")
print(f"Training accuracy: {train_acc_ft:.4f}")
print(f"Validation loss: {val_loss_ft:.4f}")
print(f"Validation accuracy: {val_acc_ft:.4f}")
print(f"Validation macro-F1: {val_f1_ft:.4f}")

In [ ]:
history_ft = [{
    'epoch': 1,
    'train_loss': train_loss_ft,
    'train_accuracy': train_acc_ft,
    'val_loss': val_loss_ft,
    'val_accuracy': val_acc_ft,
    'val_macro_f1': val_f1_ft
}]

for epoch in range(2, 6):
    train_loss_ft, train_acc_ft, train_f1_ft, _, _ = run_epoch(
        finetune_model, train_loader, criterion, optimizer
    )

    val_loss_ft, val_acc_ft, val_f1_ft, val_true_ft, val_pred_ft = run_epoch(
        finetune_model, val_loader, criterion
    )

    history_ft.append({
        'epoch': epoch,
        'train_loss': train_loss_ft,
        'train_accuracy': train_acc_ft,
        'val_loss': val_loss_ft,
        'val_accuracy': val_acc_ft,
        'val_macro_f1': val_f1_ft
    })

    print(
        f"Epoch {epoch}/5 | "
        f"Train loss: {train_loss_ft:.4f} | "
        f"Val loss: {val_loss_ft:.4f} | "
        f"Val accuracy: {val_acc_ft:.4f} | "
        f"Val macro-F1: {val_f1_ft:.4f}"
    )

In [ ]:
from pathlib import Path

pilot_path = Path(
    '/content/drive/MyDrive/dataset/'
    'resnet50_tomato_pilot_epoch1.pth'
)

print("Pilot checkpoint exists:", pilot_path.is_file())

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models
import pandas as pd

# Reload the original pilot checkpoint
checkpoint = torch.load(
    '/content/drive/MyDrive/dataset/resnet50_tomato_pilot_epoch1.pth',
    map_location=device,
    weights_only=True
)

best_model = models.resnet50(weights=None)
best_model.fc = nn.Linear(
    best_model.fc.in_features, len(class_to_idx)
)
best_model.load_state_dict(checkpoint['model_state_dict'])
best_model = best_model.to(device)

# Fine-tune the entire network
for param in best_model.parameters():
    param.requires_grad = True

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(
    best_model.parameters(),
    lr=1e-5,
    weight_decay=1e-4
)

best_f1 = -1.0
history = []

best_path = (
    '/content/drive/MyDrive/dataset/'
    'resnet50_tomato_best_finetuned.pth'
)

for epoch in range(1, 6):
    tr_loss, tr_acc, tr_f1, _, _ = run_epoch(
        best_model, train_loader, criterion, optimizer
    )

    va_loss, va_acc, va_f1, _, _ = run_epoch(
        best_model, val_loader, criterion
    )

    history.append({
        'epoch': epoch,
        'train_loss': tr_loss,
        'train_accuracy': tr_acc,
        'val_loss': va_loss,
        'val_accuracy': va_acc,
        'val_macro_f1': va_f1
    })

    if va_f1 > best_f1:
        best_f1 = va_f1

        torch.save({
            'epoch': epoch,
            'model_state_dict': best_model.state_dict(),
            'class_to_idx': class_to_idx,
            'val_loss': va_loss,
            'val_accuracy': va_acc,
            'val_macro_f1': va_f1
        }, best_path)

        print(f"Saved new best checkpoint at epoch {epoch}.")

    print(
        f"Epoch {epoch}/5 | Train loss: {tr_loss:.4f} | "
        f"Val loss: {va_loss:.4f} | Val accuracy: {va_acc:.4f} | "
        f"Val macro-F1: {va_f1:.4f}"
    )

pd.DataFrame(history).to_csv(
    '/content/drive/MyDrive/dataset/'
    'resnet50_tomato_finetuning_history.csv',
    index=False
)

print("\nBest validation macro-F1:", round(best_f1, 4))
print("Best checkpoint:", best_path)

In [ ]:
import torch
from pathlib import Path

best_path = Path(
    '/content/drive/MyDrive/dataset/'
    'resnet50_tomato_best_finetuned.pth'
)

print("Checkpoint exists:", best_path.is_file())

checkpoint = torch.load(
    best_path,
    map_location=device,
    weights_only=True
)

print("Selected epoch:", checkpoint['epoch'])
print("Validation accuracy:", round(checkpoint['val_accuracy'], 4))
print("Validation macro-F1:", round(checkpoint['val_macro_f1'], 4))

In [ ]:
import torch
from torchvision import models
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
import pandas as pd

# Rebuild the model and load the selected checkpoint
test_model = models.resnet50(weights=None)
test_model.fc = nn.Linear(
    test_model.fc.in_features, len(class_to_idx)
)

test_checkpoint = torch.load(
    '/content/drive/MyDrive/dataset/resnet50_tomato_best_finetuned.pth',
    map_location=device,
    weights_only=True
)

test_model.load_state_dict(test_checkpoint['model_state_dict'])
test_model = test_model.to(device)
test_model.eval()

# Evaluate once on the test set
test_true, test_pred = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = test_model(images)

        test_true.extend(labels.tolist())
        test_pred.extend(outputs.argmax(dim=1).cpu().tolist())

class_names = [
    name for name, idx in sorted(
        class_to_idx.items(), key=lambda item: item[1]
    )
]

print("Test accuracy:", round(accuracy_score(test_true, test_pred), 4))
print("Test macro-F1:", round(
    f1_score(test_true, test_pred, average='macro', zero_division=0), 4
))

print("\nPer-class test results:")
print(classification_report(
    test_true,
    test_pred,
    labels=list(range(len(class_names))),
    target_names=class_names,
    zero_division=0
))

print("Confusion matrix (rows = actual, columns = predicted):")
print(confusion_matrix(
    test_true, test_pred, labels=list(range(len(class_names)))
))

In [ ]:
import pandas as pd

test_result = pd.DataFrame([{
    'crop': 'Tomato',
    'model': 'ResNet50',
    'experiment': 'Fine-tuned, best validation macro-F1 checkpoint',
    'selected_epoch': test_checkpoint['epoch'],
    'validation_accuracy': test_checkpoint['val_accuracy'],
    'validation_macro_f1': test_checkpoint['val_macro_f1'],
    'test_accuracy': accuracy_score(test_true, test_pred),
    'test_macro_f1': f1_score(
        test_true, test_pred,
        average='macro', zero_division=0
    ),
    'test_images': len(test_true)
}])

results_path = (
    '/content/drive/MyDrive/dataset/'
    'tomato_baseline_results.csv'
)

test_result.to_csv(results_path, index=False)

print("Test results saved to:", results_path)
display(test_result)

In [ ]:
import torch

print("GPU available:", torch.cuda.is_available())
print("Training loader available:", 'train_loader' in globals())
print("Validation loader available:", 'val_loader' in globals())
print("Test loader available:", 'test_loader' in globals())
print("Class mapping available:", 'class_to_idx' in globals())

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load pretrained ImageNet weights
weights_eff = models.EfficientNet_B0_Weights.DEFAULT
efficientnet = models.efficientnet_b0(weights=weights_eff)

# Replace the final classifier for our five tomato classes
in_features = efficientnet.classifier[1].in_features
efficientnet.classifier[1] = nn.Linear(
    in_features, len(class_to_idx)
)

efficientnet = efficientnet.to(device)

print("Model: EfficientNet-B0")
print("Device:", device)
print("Output classes:", efficientnet.classifier[1].out_features)
print("Input size:", weights_eff.transforms().crop_size)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# Keep the pretrained backbone frozen for the pilot phase
for param in efficientnet.parameters():
    param.requires_grad = False

# Train only the final classification layer first
for param in efficientnet.classifier.parameters():
    param.requires_grad = True

criterion_eff = nn.CrossEntropyLoss()

optimizer_eff = optim.AdamW(
    efficientnet.classifier.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

print("EfficientNet-B0 training setup ready.")
print("Trainable parameters:",
      sum(p.numel() for p in efficientnet.parameters()
          if p.requires_grad))
print("Learning rate: 0.001")

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

train_loss_eff, train_acc_eff, train_f1_eff, _, _ = run_epoch(
    efficientnet, train_loader, criterion_eff, optimizer_eff
)

val_loss_eff, val_acc_eff, val_f1_eff, val_true_eff, val_pred_eff = run_epoch(
    efficientnet, val_loader, criterion_eff
)

print(f"Training loss: {train_loss_eff:.4f}")
print(f"Training accuracy: {train_acc_eff:.4f}")
print(f"Validation loss: {val_loss_eff:.4f}")
print(f"Validation accuracy: {val_acc_eff:.4f}")
print(f"Validation macro-F1: {val_f1_eff:.4f}")

In [ ]:
# Unfreeze all EfficientNet-B0 parameters
for param in efficientnet.parameters():
    param.requires_grad = True

optimizer_eff = optim.AdamW(
    efficientnet.parameters(),
    lr=1e-5,
    weight_decay=1e-4
)

print("EfficientNet-B0 ready for fine-tuning.")
print("Trainable parameters:",
      sum(p.numel() for p in efficientnet.parameters()
          if p.requires_grad))
print("Learning rate: 0.00001")

In [ ]:
train_loss_eff, train_acc_eff, train_f1_eff, _, _ = run_epoch(
    efficientnet, train_loader, criterion_eff, optimizer_eff
)

val_loss_eff, val_acc_eff, val_f1_eff, val_true_eff, val_pred_eff = run_epoch(
    efficientnet, val_loader, criterion_eff
)

print(f"Training loss: {train_loss_eff:.4f}")
print(f"Training accuracy: {train_acc_eff:.4f}")
print(f"Validation loss: {val_loss_eff:.4f}")
print(f"Validation accuracy: {val_acc_eff:.4f}")
print(f"Validation macro-F1: {val_f1_eff:.4f}")

In [ ]:
import torch
import pandas as pd

best_eff_f1 = val_f1_eff
eff_history = [{
    'epoch': 1,
    'train_loss': train_loss_eff,
    'train_accuracy': train_acc_eff,
    'val_loss': val_loss_eff,
    'val_accuracy': val_acc_eff,
    'val_macro_f1': val_f1_eff
}]

best_eff_path = (
    '/content/drive/MyDrive/dataset/'
    'efficientnet_b0_tomato_best_finetuned.pth'
)

# Save epoch 1 as the current best
torch.save({
    'epoch': 1,
    'model_state_dict': efficientnet.state_dict(),
    'class_to_idx': class_to_idx,
    'val_loss': val_loss_eff,
    'val_accuracy': val_acc_eff,
    'val_macro_f1': val_f1_eff
}, best_eff_path)

for epoch in range(2, 6):
    tr_loss, tr_acc, tr_f1, _, _ = run_epoch(
        efficientnet, train_loader, criterion_eff, optimizer_eff
    )

    va_loss, va_acc, va_f1, _, _ = run_epoch(
        efficientnet, val_loader, criterion_eff
    )

    eff_history.append({
        'epoch': epoch,
        'train_loss': tr_loss,
        'train_accuracy': tr_acc,
        'val_loss': va_loss,
        'val_accuracy': va_acc,
        'val_macro_f1': va_f1
    })

    if va_f1 > best_eff_f1:
        best_eff_f1 = va_f1

        torch.save({
            'epoch': epoch,
            'model_state_dict': efficientnet.state_dict(),
            'class_to_idx': class_to_idx,
            'val_loss': va_loss,
            'val_accuracy': va_acc,
            'val_macro_f1': va_f1
        }, best_eff_path)

        print(f"Saved new best checkpoint at epoch {epoch}.")

    print(
        f"Epoch {epoch}/5 | Train loss: {tr_loss:.4f} | "
        f"Val loss: {va_loss:.4f} | Val accuracy: {va_acc:.4f} | "
        f"Val macro-F1: {va_f1:.4f}"
    )

pd.DataFrame(eff_history).to_csv(
    '/content/drive/MyDrive/dataset/'
    'efficientnet_b0_tomato_finetuning_history.csv',
    index=False
)

print("\nBest validation macro-F1:", round(best_eff_f1, 4))
print("Best checkpoint:", best_eff_path)

In [ ]:
import torch
import torch.nn as nn
from torchvision import models
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Rebuild EfficientNet-B0 with five output classes
test_eff_model = models.efficientnet_b0(weights=None)
in_features = test_eff_model.classifier[1].in_features
test_eff_model.classifier[1] = nn.Linear(
    in_features, len(class_to_idx)
)

# Load the best validation checkpoint
eff_checkpoint = torch.load(
    '/content/drive/MyDrive/dataset/'
    'efficientnet_b0_tomato_best_finetuned.pth',
    map_location=device,
    weights_only=True
)

test_eff_model.load_state_dict(
    eff_checkpoint['model_state_dict']
)
test_eff_model = test_eff_model.to(device)
test_eff_model.eval()

# Evaluate the held-out test set
test_true_eff, test_pred_eff = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = test_eff_model(images)

        test_true_eff.extend(labels.tolist())
        test_pred_eff.extend(
            outputs.argmax(dim=1).cpu().tolist()
        )

class_names = [
    name for name, idx in sorted(
        class_to_idx.items(), key=lambda item: item[1]
    )
]

print("Selected epoch:", eff_checkpoint['epoch'])
print("Test accuracy:",
      round(accuracy_score(test_true_eff, test_pred_eff), 4))
print("Test macro-F1:",
      round(f1_score(
          test_true_eff, test_pred_eff,
          average='macro', zero_division=0
      ), 4))

print("\nPer-class test results:")
print(classification_report(
    test_true_eff,
    test_pred_eff,
    labels=list(range(len(class_names))),
    target_names=class_names,
    zero_division=0
))

print("Confusion matrix:")
print(confusion_matrix(
    test_true_eff,
    test_pred_eff,
    labels=list(range(len(class_names)))
))

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score

results_path = Path(
    '/content/drive/MyDrive/dataset/tomato_baseline_results.csv'
)

efficientnet_result = pd.DataFrame([{
    'crop': 'Tomato',
    'model': 'EfficientNet-B0',
    'experiment': 'Fine-tuned, best validation macro-F1 checkpoint',
    'selected_epoch': eff_checkpoint['epoch'],
    'validation_accuracy': eff_checkpoint['val_accuracy'],
    'validation_macro_f1': eff_checkpoint['val_macro_f1'],
    'test_accuracy': accuracy_score(test_true_eff, test_pred_eff),
    'test_macro_f1': f1_score(
        test_true_eff, test_pred_eff,
        average='macro', zero_division=0
    ),
    'test_images': len(test_true_eff)
}])

if results_path.exists():
    existing = pd.read_csv(results_path)
    existing = existing[existing['model'] != 'EfficientNet-B0']
    results = pd.concat(
        [existing, efficientnet_result],
        ignore_index=True
    )
else:
    results = efficientnet_result

results.to_csv(results_path, index=False)

print("Results saved:", results_path)
display(results[['crop', 'model', 'selected_epoch',
                 'validation_accuracy', 'validation_macro_f1',
                 'test_accuracy', 'test_macro_f1', 'test_images']])

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load pretrained DenseNet121
weights_dense = models.DenseNet121_Weights.DEFAULT
densenet = models.densenet121(weights=weights_dense)

# Replace the final classifier for five tomato classes
in_features = densenet.classifier.in_features
densenet.classifier = nn.Linear(
    in_features, len(class_to_idx)
)

densenet = densenet.to(device)

print("Model: DenseNet121")
print("Device:", device)
print("Output classes:", densenet.classifier.out_features)
print("Input size:", weights_dense.transforms().crop_size)

In [ ]:
import torch.nn as nn
import torch.optim as optim

# Freeze the pretrained backbone
for param in densenet.parameters():
    param.requires_grad = False

# Unfreeze only the final classifier
for param in densenet.classifier.parameters():
    param.requires_grad = True

criterion_dense = nn.CrossEntropyLoss()

optimizer_dense = optim.AdamW(
    densenet.classifier.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

print("DenseNet121 pilot setup ready.")
print(
    "Trainable parameters:",
    sum(p.numel() for p in densenet.parameters()
        if p.requires_grad)
)
print("Learning rate: 0.001")

In [ ]:
train_loss_dense, train_acc_dense, train_f1_dense, _, _ = run_epoch(
    densenet, train_loader, criterion_dense, optimizer_dense
)

val_loss_dense, val_acc_dense, val_f1_dense, val_true_dense, val_pred_dense = run_epoch(
    densenet, val_loader, criterion_dense
)

print(f"Training loss: {train_loss_dense:.4f}")
print(f"Training accuracy: {train_acc_dense:.4f}")
print(f"Validation loss: {val_loss_dense:.4f}")
print(f"Validation accuracy: {val_acc_dense:.4f}")
print(f"Validation macro-F1: {val_f1_dense:.4f}")

In [ ]:
# Unfreeze all DenseNet121 parameters
for param in densenet.parameters():
    param.requires_grad = True

optimizer_dense = optim.AdamW(
    densenet.parameters(),
    lr=1e-5,
    weight_decay=1e-4
)

print("DenseNet121 ready for fine-tuning.")
print(
    "Trainable parameters:",
    sum(p.numel() for p in densenet.parameters()
        if p.requires_grad)
)
print("Learning rate: 0.00001")

In [ ]:
train_loss_dense, train_acc_dense, train_f1_dense, _, _ = run_epoch(
    densenet, train_loader, criterion_dense, optimizer_dense
)

val_loss_dense, val_acc_dense, val_f1_dense, val_true_dense, val_pred_dense = run_epoch(
    densenet, val_loader, criterion_dense
)

print(f"Training loss: {train_loss_dense:.4f}")
print(f"Training accuracy: {train_acc_dense:.4f}")
print(f"Validation loss: {val_loss_dense:.4f}")
print(f"Validation accuracy: {val_acc_dense:.4f}")
print(f"Validation macro-F1: {val_f1_dense:.4f}")

In [ ]:
import torch
import pandas as pd

best_dense_f1 = val_f1_dense

dense_history = [{
    'epoch': 1,
    'train_loss': train_loss_dense,
    'train_accuracy': train_acc_dense,
    'val_loss': val_loss_dense,
    'val_accuracy': val_acc_dense,
    'val_macro_f1': val_f1_dense
}]

best_dense_path = (
    '/content/drive/MyDrive/dataset/'
    'densenet121_tomato_best_finetuned.pth'
)

# Save the current epoch 1 checkpoint
torch.save({
    'epoch': 1,
    'model_state_dict': densenet.state_dict(),
    'class_to_idx': class_to_idx,
    'val_loss': val_loss_dense,
    'val_accuracy': val_acc_dense,
    'val_macro_f1': val_f1_dense
}, best_dense_path)

for epoch in range(2, 6):
    tr_loss, tr_acc, tr_f1, _, _ = run_epoch(
        densenet, train_loader, criterion_dense, optimizer_dense
    )

    va_loss, va_acc, va_f1, _, _ = run_epoch(
        densenet, val_loader, criterion_dense
    )

    dense_history.append({
        'epoch': epoch,
        'train_loss': tr_loss,
        'train_accuracy': tr_acc,
        'val_loss': va_loss,
        'val_accuracy': va_acc,
        'val_macro_f1': va_f1
    })

    if va_f1 > best_dense_f1:
        best_dense_f1 = va_f1

        torch.save({
            'epoch': epoch,
            'model_state_dict': densenet.state_dict(),
            'class_to_idx': class_to_idx,
            'val_loss': va_loss,
            'val_accuracy': va_acc,
            'val_macro_f1': va_f1
        }, best_dense_path)

        print(f"Saved new best checkpoint at epoch {epoch}.")

    print(
        f"Epoch {epoch}/5 | Train loss: {tr_loss:.4f} | "
        f"Val loss: {va_loss:.4f} | Val accuracy: {va_acc:.4f} | "
        f"Val macro-F1: {va_f1:.4f}"
    )

pd.DataFrame(dense_history).to_csv(
    '/content/drive/MyDrive/dataset/'
    'densenet121_tomato_finetuning_history.csv',
    index=False
)

print("\nBest validation macro-F1:", round(best_dense_f1, 4))
print("Best checkpoint:", best_dense_path)

In [ ]:
import torch
import torch.nn as nn
from torchvision import models
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Recreate DenseNet121 for five classes
test_dense_model = models.densenet121(weights=None)
test_dense_model.classifier = nn.Linear(
    test_dense_model.classifier.in_features,
    len(class_to_idx)
)

# Load the best validation checkpoint
dense_checkpoint = torch.load(
    '/content/drive/MyDrive/dataset/'
    'densenet121_tomato_best_finetuned.pth',
    map_location=device,
    weights_only=True
)

test_dense_model.load_state_dict(
    dense_checkpoint['model_state_dict']
)
test_dense_model = test_dense_model.to(device)
test_dense_model.eval()

# Evaluate the held-out test set
test_true_dense, test_pred_dense = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = test_dense_model(images)

        test_true_dense.extend(labels.tolist())
        test_pred_dense.extend(
            outputs.argmax(dim=1).cpu().tolist()
        )

class_names = [
    name for name, idx in sorted(
        class_to_idx.items(), key=lambda item: item[1]
    )
]

print("Selected epoch:", dense_checkpoint['epoch'])
print("Test accuracy:",
      round(accuracy_score(test_true_dense, test_pred_dense), 4))
print("Test macro-F1:",
      round(f1_score(
          test_true_dense, test_pred_dense,
          average='macro', zero_division=0
      ), 4))

print("\nPer-class test results:")
print(classification_report(
    test_true_dense,
    test_pred_dense,
    labels=list(range(len(class_names))),
    target_names=class_names,
    zero_division=0
))

print("Confusion matrix:")
print(confusion_matrix(
    test_true_dense,
    test_pred_dense,
    labels=list(range(len(class_names)))
))

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score

results_path = Path(
    '/content/drive/MyDrive/dataset/tomato_baseline_results.csv'
)

dense_result = pd.DataFrame([{
    'crop': 'Tomato',
    'model': 'DenseNet121',
    'experiment': 'Fine-tuned, best validation macro-F1 checkpoint',
    'selected_epoch': dense_checkpoint['epoch'],
    'validation_accuracy': dense_checkpoint['val_accuracy'],
    'validation_macro_f1': dense_checkpoint['val_macro_f1'],
    'test_accuracy': accuracy_score(test_true_dense, test_pred_dense),
    'test_macro_f1': f1_score(
        test_true_dense, test_pred_dense,
        average='macro', zero_division=0
    ),
    'test_images': len(test_true_dense)
}])

if results_path.exists():
    existing = pd.read_csv(results_path)
    existing = existing[existing['model'] != 'DenseNet121']
    results = pd.concat([existing, dense_result], ignore_index=True)
else:
    results = dense_result

results.to_csv(results_path, index=False)

print("Results saved:", results_path)
display(results)